# Wheeler-Kiladis Wavenumber-Frequency Spectrum
Panel plots of equatorial wave spectra for observations and CESM3 simulations.

**References**
- Wheeler, M. & Kiladis, G.N. (1999), *J. Atmos. Sci.*, 56, 374–399
- Hayashi, Y. (1971), *J. Meteor. Soc. Japan*, 49, 125–128

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import importlib
import sys, os
os.environ["DASK_TEMPORARY_DIRECTORY"] = "/glade/derecho/scratch/rneale/dask-temp"


sys.path.insert(0, os.path.dirname(os.path.abspath('kf_pan_utils.py')))
import kf_pan_utils as kfp
importlib.reload(kfp)

In [ ]:
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [ ]:
cluster = PBSCluster(
    account="P03010039",
    interface="ext",
    walltime="12:00:00",
    queue="main",
    cores=4,
    memory="32GB",
    processes=4,
    local_directory="/glade/derecho/scratch/rneale/dask-temp",
    log_directory="/glade/derecho/scratch/rneale/dask-temp",
    job_extra_directives=["-m n"],
    job_script_prologue=[
        "mkdir -p /glade/derecho/scratch/rneale/dask-temp"
    ],
)


cluster.scale(jobs=32)
client = Client(cluster)

client

## Setup Run Information

## Case specifications
Edit the cells below to select cases, variables, years, and plotting options.

In [ ]:
# ── Variable ───────────────────────────────────────────────────────────────
VAR0    = 'PRECT'      # 'PRECT', 'U850', 'U200'
vscale0 = 86400. * 1000.  # m/s → mm/day for PRECT; set 1.0 for obs or winds

# ── Spectral parameters ────────────────────────────────────────────────────
lat_bound = 15       # symmetric equatorial belt (degrees)
n_day_win = 96       # temporal window length (days)
n_day_skip = -65     # days between windows (negative = overlap)

# ── Plot limits ────────────────────────────────────────────────────────────
min_wav  = -15
max_wav  =  15
max_freq =   0.8   # cpd

# ── Anomaly mode ───────────────────────────────────────────────────────────
anom_plot = False    # if True, plot ratio to case 0 (first in list)

# ── Output ─────────────────────────────────────────────────────────────────
save_dir   = './figs_kf_pan'
fig_prefix = 'kf_pan'

In [ ]:
# ── Cases ──────────────────────────────────────────────────────────────────
# Each entry is a dict describing one case.
# 'source' can be: 'TRMM', 'GPCP', 'ERA5', 'IMERG', or a CESM case string.
# 'label'  : short string for plot titles
# 'yr0/yr1': year range
# 'freq'   : 'daily' or '3hourly'
# 'resolution': '1deg' or '0.25deg' (IMERG only)
# 'var'    : variable name in the file (ERA5 short names: 'tp', 'u', 'v')
# 'vscale' : unit scale (override global vscale0 if needed)

cases = [
    dict(source='TRMM',  label='TRMM',  yr0=2000, yr1=2009, freq='daily',   var='PRECT', vscale=1.0),
    dict(source='GPCP',  label='GPCP',  yr0=2000, yr1=2009, freq='daily',   var='PRECT', vscale=1.0),
    dict(source='b.e30_alpha08b.B1850C_LTso.ne30_t232_wgx3.315',
         label='315',   yr0=30,   yr1=59,  freq='daily',   var=VAR0,    vscale=vscale0),
    dict(source='b.e30_alpha08b.B1850C_LTso.ne30_t232_wgx3.316',
         label='316',   yr0=20,   yr1=49,  freq='daily',   var=VAR0,    vscale=vscale0),
]

# ── IMERG example (comment out / swap in as needed) ────────────────────────
# cases = [
#     dict(source='IMERG', label='IMERG 1deg', yr0=2001, yr1=2010,
#          freq='3hourly', resolution='1deg', var='precip', vscale=1.0),
#     dict(source='ERA5',  label='ERA5',       yr0=2001, yr1=2010,
#          freq='3hourly', var='tp', vscale=1.0),
# ]

case_labels = [c['label'] for c in cases]
print('Cases:', case_labels)

In [ ]:
# ── Contour levels ─────────────────────────────────────────────────────────
if VAR0 in ('U850', 'U200', 'OMEGA500'):
    fig_1_levels  = np.linspace(-1.0, 0.4, 15)
    fig_2_levels  = np.linspace(-1.0, 0.4, 15)
    fig_3a_levels = np.array([0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5,
                               1.6, 1.7, 1.8, 2.0, 2.5, 3.0, 3.5])
    fig_3b_levels = fig_3a_levels.copy()
else:  # PRECT default
    fig_1_levels  = np.linspace(-1.0, 0.4, 15)
    fig_2_levels  = fig_1_levels.copy()
    fig_3a_levels = np.array([0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.15,
                               1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.6])
    fig_3b_levels = fig_3a_levels.copy()

anom_levels = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8,
                         0.7, 0.9, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0, 3.0, 4.0, 5.0])

## Load data and compute WK spectra

In [ ]:
results = []

for ic, case in enumerate(cases):
    src    = case['source']
    lbl    = case['label']
    yr0    = case.get('yr0')
    yr1    = case.get('yr1')
    var    = case.get('var', VAR0)
    freq   = case.get('freq', 'daily')
    res    = case.get('resolution', '1deg')
    vs     = case.get('vscale', vscale0)
    lvl    = case.get('level_hPa', None)

    print(f'\n--- Loading case {ic+1}/{len(cases)}: {lbl} ---')

    x_np, lat, lon, spd = kfp.load_data(
        source=src, var=var, resolution=res, freq=freq,
        yr0=yr0, yr1=yr1, lat_bound=lat_bound, level_hPa=lvl,
    )
    print(f'  Loaded: shape={x_np.shape}, spd={spd}, lat={lat[[0,-1]]}')

    print(f'  Computing WK spectrum...')
    res_dict = kfp.compute_wk_spectrum(
        x_np, lat, spd=spd,
        n_day_win=n_day_win, n_day_skip=n_day_skip,
        vscale=vs, lat_bound=lat_bound,
    )
    res_dict['label'] = lbl
    results.append(res_dict)
    print(f'  Done. wave shape: {res_dict["wave"].shape}, freq shape: {res_dict["freq"].shape}')

## Generate dispersion curves

In [ ]:
Apzwn, Afreq = kfp.gen_dispersion_curves(
    ahe=(50., 25., 12.),
    n_wave_type=6,
    n_planetary_wave=50,
)
print('Dispersion curves computed. Apzwn shape:', Apzwn.shape)

## Plot WK panels
Five figures are produced:
1. Antisymmetric spectrum (log10)
2. Symmetric spectrum (log10)
3. Background spectrum (log10)
4. Antisymmetric / Background ratio
5. Symmetric / Background ratio

In [ ]:
figs = kfp.plot_wk_panel(
    results       = results,
    case_labels   = case_labels,
    var_name      = VAR0,
    lat_bound     = lat_bound,
    fig_1_levels  = fig_1_levels,
    fig_2_levels  = fig_2_levels,
    fig_3a_levels = fig_3a_levels,
    fig_3b_levels = fig_3b_levels,
    anom_plot     = anom_plot,
    anom_levels   = anom_levels,
    Apzwn         = Apzwn,
    Afreq         = Afreq,
    add_disp_lines= True,
    add_mjo_box   = True,
    min_wav       = min_wav,
    max_wav       = max_wav,
    max_freq      = max_freq,
    cmap          = 'RdBu_r',
    cmap_ratio    = 'RdBu_r',
    save_dir      = save_dir,
    fig_prefix    = fig_prefix + '_' + VAR0,
)
plt.show()
print('DONE')

## Quick-look: single case spectrum

In [ ]:
# Inspect a single case result in detail
ic_show = 0   # index into results list
res = results[ic_show]
wave = res['wave']
freq = res['freq']

# Subset to plot range, positive freq only
iw0 = np.searchsorted(wave, min_wav)
iw1 = np.searchsorted(wave, max_wav) + 1
if0 = np.searchsorted(freq, 0.0)
if1 = np.searchsorted(freq, max_freq) + 1

W = wave[iw0:iw1]
F = freq[if0:if1]
WW, FF = np.meshgrid(W, F)

sym_r  = (res['psumsym_nl']  / res['psumb_nl'])[iw0:iw1, if0:if1].T
asym_r = (res['psumanti_nl'] / res['psumb_nl'])[iw0:iw1, if0:if1].T

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, title in zip(axes,
                            [asym_r, sym_r],
                            ['Antisymmetric/BG', 'Symmetric/BG']):
    cf = ax.contourf(WW, FF, data, levels=fig_3a_levels, cmap='RdBu_r', extend='both')
    plt.colorbar(cf, ax=ax)
    kfp.add_hor_vert_lines(ax, min_wav, max_wav)
    if Apzwn is not None:
        kfp.add_dispersion_curves(
            ax, Apzwn, Afreq,
            plot_type='sym' if 'Sym' in title else 'asym',
            min_wav=min_wav, max_wav=max_wav, max_freq=max_freq
        )
    ax.set_xlim(min_wav, max_wav)
    ax.set_ylim(0, max_freq)
    ax.set_title(f"{case_labels[ic_show]} — {title}")
    ax.set_xlabel('Zonal Wave Number')
    ax.set_ylabel('Frequency (cpd)')

fig.tight_layout()
plt.show()